# Tutorial 2: Points on the Curve


**Audience:** Completed Tutorial 1

**Prerequisites:** Scalar arithmetic

**Learning goals:**
- Understand secp256k1 curve points and the generator G
- Perform point addition and scalar multiplication
- Serialize and deserialize points (BIP340 x-only format)


## What this notebook covers

1. The generator point G and the secp256k1 curve equation
2. Scalar multiplication: k·G
3. The point at infinity (identity element)
4. Serialization: compressed and x-only (BIP340)
5. Even-y convention and lift_x
6. Exercise: generate a keypair


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Scalar, Point, G, P, Q


## The Generator Point

G is a specific point on secp256k1 that generates the entire cyclic group.
Every point in the group can be written as k·G for some scalar k.
The curve equation is y² = x³ + 7 (mod P).


In [ ]:
print(f"G.x = {G.x}")
print(f"G.y = {G.y}")
print(f"On curve? y² mod P == (x³ + 7) mod P: {pow(G.y, 2, P) == (pow(G.x, 3, P) + 7) % P}")


## Scalar Multiplication

k·G means "add G to itself k times." This is the fundamental operation
in elliptic curve cryptography. It is easy to compute k·G given k,
but computationally infeasible to recover k given k·G.


In [ ]:
P2 = 2 * G
P3 = 3 * G
print(f"2·G = ({P2.x}, {P2.y})")
print(f"3·G = ({P3.x}, {P3.y})")
print(f"G + G == 2·G? {G + G == P2}")
print(f"G + G + G == 3·G? {G + G + G == P3}")


## Point at Infinity

The point at infinity is the identity element for curve addition.
Adding it to any point returns that point unchanged, like adding zero.


In [ ]:
O = Point()  # Point at infinity
print(f"Point at infinity: x={O.x}, y={O.y}")
print(f"G + O == G? {G + O == G}")
print(f"G + (-G) == O? {(G + (-G)).is_zero()}")


## Serialization

Points can be serialized in two formats:
- **Compressed** (33 bytes): prefix byte (02/03) + x-coordinate
- **X-only** (32 bytes): just the x-coordinate (BIP340 format)


In [ ]:
compressed = G.to_bytes_compressed()
xonly = G.to_bytes_xonly()
print(f"Compressed ({len(compressed)} bytes): {compressed.hex()[:20]}...")
print(f"X-only ({len(xonly)} bytes): {xonly.hex()[:20]}...")

# Roundtrip
G2 = Point.from_bytes_compressed(compressed.hex())
print(f"Roundtrip: {G == G2}")


## Even-y Convention

BIP340 uses x-only public keys. For any x-coordinate on the curve, there
are exactly two valid y values (y and P - y), one even and one odd.
BIP340 always picks the even one.


In [ ]:
print(f"G has even y? {G.has_even_y()}")
neg_G = -G
print(f"-G has even y? {neg_G.has_even_y()}")
print(f"Same x? {G.x == neg_G.x}")
print(f"y values sum to P? {(G.y + neg_G.y) % P == 0}")


## lift_x

Given just an x-coordinate, `lift_x` recovers the full point by computing
the square root of x³ + 7 mod P and choosing the even y (BIP340 convention).


In [ ]:
x_bytes = G.to_bytes_xonly()
G_lifted = Point.lift_x(int.from_bytes(x_bytes, "big"))
print(f"lift_x recovered G? {G_lifted == G}")
print(f"Recovered point has even y? {G_lifted.has_even_y()}")


## Exercise

Generate a keypair: random scalar = private key, scalar · G = public key.
Verify the public key is on the curve.


In [ ]:
private_key = Scalar.random()
public_key = int(private_key) * G
on_curve = pow(public_key.y, 2, P) == (pow(public_key.x, 3, P) + 7) % P
print(f"Private key: {int(private_key)}")
print(f"Public key: ({public_key.x}, ...)")
print(f"On curve? {on_curve}")


## Pitfalls and Extensions

**Pitfall:** Point addition is NOT coordinate addition.
(x₁ + x₂, y₁ + y₂) is generally NOT on the curve. Curve addition uses
the chord-and-tangent rule (see `point.py` for the formulas).

**Extension:** The discrete log problem: given P = k·G, finding k is
computationally infeasible. This is the foundation of all public key
cryptography on elliptic curves.
